<a href="https://colab.research.google.com/github/HarshithReddy01/Algorithms-Practice/blob/master/Redosprojectlegit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install redos-analyzer

In [2]:
from redos_analyzer import analyze

patterns = [
    (r'(a+)+',          'nested quantifier classic'),
    (r'([a-zA-Z]+)*',   'nested quantifier on letters'),
    (r'(a|aa)+',        'overlapping alternation'),
    (r'(a|a?)+',        'nullable branch'),
    (r'(:?\w|:|\.)+',   'the pytest typo pattern'),
]

for pattern, label in patterns:
    warnings = analyze(pattern)
    print(f"\n{label}")
    print(f"Pattern: {pattern}")
    print(f"Warnings: {len(warnings)}")
    for w in warnings:
        print(f"  {w.kind} — {w.message[:80]}")


nested quantifier classic
Pattern: (a+)+
Warnings: 1
  NESTED_QUANTIFIER — Repeating quantifier '+' on 'a' is nested inside the outer repeating quantifier.

nested quantifier on letters
Pattern: ([a-zA-Z]+)*
Warnings: 1
  NESTED_QUANTIFIER — Repeating quantifier '+' on '[a-zA-Z]' is nested inside the outer repeating quan

overlapping alternation
Pattern: (a|aa)+
Warnings: 2
  NULLABLE_BRANCH_IN_QUANTIFIER — Alternative(s) [''] inside quantifier '+' can match the empty string.  The group
  OVERLAPPING_ALTERNATION — Branch '' is a literal prefix of 'a' (or vice-versa) inside quantifier '+'.  The

nullable branch
Pattern: (a|a?)+
Warnings: 2
  NULLABLE_BRANCH_IN_QUANTIFIER — Alternative(s) ['a?'] inside quantifier '+' can match the empty string.  The gro
  OVERLAPPING_ALTERNATION — Branches 'a' and 'a?' share overlapping first-character sets inside quantifier '

the pytest typo pattern
Pattern: (:?\w|:|\.)+
Warnings: 4
  LIKELY_TYPO — Found '(:?' - this looks like a typo for '(?:' (non-c

In [3]:
safe_patterns = [
    (r'(a|b)+',         'non overlapping alternation'),
    (r'(ab|ac)+',       'different second char'),
    (r'(a+)?',          'quantifier inside optional'),
    (r'\d{3}-\d{4}',    'phone number'),
    (r'https?://\S+',   'url pattern'),
]

print("Safe patterns — all should show 0 warnings\n")
for pattern, label in safe_patterns:
    warnings = analyze(pattern)
    status = "PASS" if len(warnings) == 0 else "FALSE POSITIVE"
    print(f"{status}  {label}  ({pattern})")

Safe patterns — all should show 0 warnings

PASS  non overlapping alternation  ((a|b)+)
PASS  different second char  ((ab|ac)+)
PASS  quantifier inside optional  ((a+)?)
PASS  phone number  (\d{3}-\d{4})
PASS  url pattern  (https?://\S+)


In [4]:
from redos_analyzer import analyze, suggest_fix

pattern = r'([a-zA-Z]+)*'

print(f"Original pattern: {pattern}")
print()

warnings = analyze(pattern)
print(f"Warnings found: {len(warnings)}")
print(f"Warning type: {warnings[0].kind}")
print(f"Detail: {warnings[0].message}")
print()

fix = suggest_fix(pattern, warnings[0])
print(f"Fixed pattern:  {fix['fixed']}")
print(f"Fix type:       {fix['fix_type']}")
print(f"Verified:       {fix['verified']}")
print(f"Explanation:    {fix['explanation']}")

Original pattern: ([a-zA-Z]+)*

Warnings found: 1
Warning type: NESTED_QUANTIFIER
Detail: Repeating quantifier '+' on '[a-zA-Z]' is nested inside the outer repeating quantifier.  On a failing match the engine must explore an exponential number of ways to partition the input across the two quantifiers.

Fixed pattern:  ((?>[a-zA-Z]+))*
Fix type:       atomic_group
Verified:       True
Explanation:    Dangerous repeating group(s) wrapped with atomic-group syntax (?>...). Warning class: NESTED_QUANTIFIER. Atomic groups prevent the engine from reconsidering what the group matched once it has committed, eliminating exponential backtracking. NOTE: sre_parse normalises escape sequences (\s→[\s] etc.); the fixed pattern is semantically equivalent but may look different. Capturing groups are preserved as ((?>...)) so group numbers and named back-references remain valid.


In [5]:
import re
import time

original = r'([a-zA-Z]+)*'
fixed    = fix['fixed']

evil_input = 'a' * 22 + '!'

print("Testing original pattern against evil input...")
compiled_original = re.compile(original + '$')
start = time.perf_counter()
compiled_original.match(evil_input)
original_time = time.perf_counter() - start
print(f"Original: {original_time:.4f}s")

print("\nTesting fixed pattern against evil input...")
compiled_fixed = re.compile(fixed + '$')
start = time.perf_counter()
compiled_fixed.match(evil_input)
fixed_time = time.perf_counter() - start
print(f"Fixed:    {fixed_time:.6f}s")

if original_time > 0 and fixed_time > 0:
    speedup = original_time / fixed_time
    print(f"\nSpeedup:  {speedup:.0f}x faster")
else:
    print("\nFixed pattern too fast to measure meaningfully")

Testing original pattern against evil input...
Original: 0.3060s

Testing fixed pattern against evil input...
Fixed:    0.000111s

Speedup:  2754x faster


In [6]:
from redos_analyzer import verify_fix

result = verify_fix(original, fixed)

print(f"Semantics preserved: {result['semantics_preserved']}")
print(f"Speedup factor:      {result['speedup_factor']}x")
print(f"Evil input tested:   {result['evil_input_tested']!r}")

if result.get('semantic_mismatches'):
    print(f"Mismatches: {result['semantic_mismatches']}")
else:
    print("Zero mismatches on all test inputs")

Semantics preserved: True
Speedup factor:      1.5x
Evil input tested:   'aaaaaaaaaaaaaaaaaaaaaa!'
Zero mismatches on all test inputs


In [7]:
# This is the actual pattern from pytest expression.py
# The developer wrote (:? instead of (?:
# (:? means: capturing group where colon is optional
# (?:  means: non-capturing group

typo_pattern = r'(:?\w|:|\.|\[|\]|\\|/)+'

warnings = analyze(typo_pattern)

print(f"Warnings found: {len(warnings)}")
for w in warnings:
    print(f"\nKind: {w.kind}")
    print(f"Message: {w.message[:120]}")

print("\nThe LIKELY_TYPO warning means the developer")
print("almost certainly meant (?:  not  (:?")
print("These look similar but do completely different things")

Warnings found: 8

Kind: LIKELY_TYPO
Message: Found '(:?' - this looks like a typo for '(?:' (non-capturing group). '(:?' creates a capturing group that optionally ma

Kind: NULLABLE_BRANCH_IN_QUANTIFIER
Message: Alternative(s) [':?[\\w]'] inside quantifier '+' can match the empty string.  The group can iterate without consuming in

Kind: OVERLAPPING_ALTERNATION
Message: Branches ':?[\w]' and ':' share overlapping first-character sets inside quantifier '+'. The same input may be matched in

Kind: OVERLAPPING_ALTERNATION
Message: Branches ':?[\w]' and '\.' share overlapping first-character sets inside quantifier '+'. The same input may be matched i

Kind: OVERLAPPING_ALTERNATION
Message: Branches ':?[\w]' and '\[' share overlapping first-character sets inside quantifier '+'. The same input may be matched i

Kind: OVERLAPPING_ALTERNATION
Message: Branches ':?[\w]' and '\]' share overlapping first-character sets inside quantifier '+'. The same input may be matched i

Kind: OVERLAPPING_ALT

In [8]:
# Real pattern from sqlalchemy dialects/sqlite/base.py
sqlalchemy_pattern = r'create table .*?\((.*)\)(?:\s*,?\s*(?:WITHOUT\s+ROWID|STRICT))*$'

print(f"Pattern: {sqlalchemy_pattern}")
print()

warnings = analyze(sqlalchemy_pattern)
print(f"Warnings found: {len(warnings)}")
for w in warnings:
    print(f"\nKind:     {w.kind}")
    print(f"Severity: NESTED_QUANTIFIER is HIGH")
    print(f"Detail:   {w.message[:100]}")

    fix = suggest_fix(sqlalchemy_pattern, w)
    print(f"\nFix type: {fix['fix_type']}")
    if fix['fixed']:
        print(f"Fixed:    {fix['fixed'][:80]}")
        print(f"Verified: {fix['verified']}")
    else:
        print("Manual review required for this pattern")
    break

Pattern: create table .*?\((.*)\)(?:\s*,?\s*(?:WITHOUT\s+ROWID|STRICT))*$

Warnings found: 3

Kind:     NESTED_QUANTIFIER
Severity: NESTED_QUANTIFIER is HIGH
Detail:   Repeating quantifier '*' on '[\s]' is nested inside the outer repeating quantifier.  On a failing ma

Fix type: atomic_group
Fixed:    create table .*?\((.*)\)(?>[\s]*,?[\s]*(?:WITHOUT[\s]+ROWID|STRICT))*$
Verified: True
